#30th Session

In [0]:
data4 = [(1, "raj"), (2, "ravi"), (3, "sai"), (5, "rani")]
cust = spark.createDataFrame(data4, ["id", "name"])
cust.display()

data3 = [(1, "mouse"), (3, "mobile"), (7, "laptop")]
prod = spark.createDataFrame(data3, ["id", "product"])
prod.display()

id,name
1,raj
2,ravi
3,sai
5,rani


id,product
1,mouse
3,mobile
7,laptop


In [0]:
df_innerj = cust.join(prod,"id","inner")
df_innerj.display()

id,name,product
1,raj,mouse
3,sai,mobile


In [0]:
df_leftj = cust.join(prod,"id","left")
df_leftj.display()

id,name,product
1,raj,mouse
2,ravi,null
3,sai,mobile
5,rani,null


In [0]:
df_rightj = cust.join(prod,"id","right")
df_rightj.display()

id,name,product
1,raj,mouse
3,sai,mobile
7,null,laptop


In [0]:
df_fullj = cust.join(prod,"id","full")
df_fullj.display()

id,name,product
1,raj,mouse
2,ravi,null
3,sai,mobile
5,rani,null
7,null,laptop


In [0]:
df_leftantij = cust.join(prod,"id","left_anti")
df_leftantij.display()

id,name
2,ravi
5,rani


In [0]:
df_leftantij = prod.join( cust,"id","left_anti")
df_leftantij.display()

id,product
7,laptop


In [0]:
df_fullj = cust.join(prod,"id","full").filter("name is null")
df_fullj.display()

id,name,product
7,null,laptop


In [0]:
df_crossj = cust.crossJoin(prod)
df_crossj.display()

id,name,id,product
1,raj,1,mouse
1,raj,3,mobile
1,raj,7,laptop
2,ravi,1,mouse
2,ravi,3,mobile
2,ravi,7,laptop
3,sai,1,mouse
3,sai,3,mobile
3,sai,7,laptop
5,rani,1,mouse


In [0]:
data4 = [(1, "raj"), (2, "ravi"), (3, "sai"), (5, "rani")]
cust = spark.createDataFrame(data4, ["id", "name"])
cust.display()

data3 = [(1, "mouse"), (3, "mobile"), (7, "laptop")]
prod = spark.createDataFrame(data3, ["cid", "product"])
prod.display()

id,name
1,raj
2,ravi
3,sai
5,rani


cid,product
1,mouse
3,mobile
7,laptop


In [0]:
df_innerj = cust.join(prod, cust["id"] == prod["cid"] ,"inner").drop("cid")
df_innerj.display()

id,name,product
1,raj,mouse
3,sai,mobile


In [0]:
# Sample data from the image
data = [
    ("A", "AA"),
    ("B", "BB"),
    ("C", "CC"),
    ("AA", "AAA"),
    ("BB", "BBB"),
    ("CC", "CCC"),
]

# Define columns
columns = ["child", "parent"]

# Create DataFrame
df = spark.createDataFrame(data, schema=columns)
df.createOrReplaceTempView("tab")
# Display result
df.display()

child,parent
A,AA
B,BB
C,CC
AA,AAA
BB,BBB
CC,CCC


In [0]:
%sql
select b.child,a.child parent ,a.parent grand_parent from tab a join tab b on a.child = b.parent

child,parent,grand_parent
A,AA,AAA
B,BB,BBB
C,CC,CCC


In [0]:
df1=df
df2 =df.withColumnRenamed("child","child1").withColumnRenamed("parent","parent1")
df_join = df1.join(df2, df1["child"] == df2["parent1"], "inner").selectExpr("child1 as child", "child as parent", "parent as grand_parent")
df_join.display()

child,parent,grand_parent
A,AA,AAA
B,BB,BBB
C,CC,CCC


In [0]:
# 1. Left Dataframe (Source)
source_data = [(1, "A"), (2, "B"), (3, "C"), (4, "D")]
source_cols = ["id", "name"]
df_src = spark.createDataFrame(source_data, schema=source_cols)

# 2. Right Dataframe (Target)
target_data = [(1, "A"), (2, "B"), (4, "X"), (5, "F")]
target_cols = ["id1", "name1"]
df_tgt = spark.createDataFrame(target_data, schema=target_cols)

# Display both data frames
print("Source DataFrame:")
df_src.display()

print("Target DataFrame:")
df_tgt.display()


Source DataFrame:


id,name
1,A
2,B
3,C
4,D


Target DataFrame:


id1,name1
1,A
2,B
4,X
5,F


In [0]:
from pyspark.sql.functions import *

df_join= df_src.join(df_tgt, df_src["id"] == df_tgt["id1"], "full").filter("coalesce(name,'') <> coalesce(name1,'')").withColumn("comment",expr("case when id1 is null then 'New in Source' when id is null then 'New in Target' when name <> name1 then 'Mismatch'  else '' end")).selectExpr("coalesce(id,id1) as id", "comment")
df_join.display()

id,comment
3,New in Source
4,Mismatch
5,New in Target


In [0]:
from pyspark.sql.functions import *

df_join = (
    df_src.join(df_tgt, df_src["id"] == df_tgt["id1"], "full")
    .selectExpr(
        "coalesce(id,id1) as id",
        "case when name = name1 then 'Matched'  when name <> name1 then 'Mismatch' when name is null then 'New in Source'  when name1 is null then 'New in Target'  end as comment",
    )
    .filter("comment <> 'Matched'")
)
df_join.display()

id,comment
3,New in Target
4,mismatch
5,New in Source
